In [16]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time

In [17]:
# Load API key
load_dotenv("../../.env")
api_key = os.getenv("NOAA_API_KEY")

headers = {"token": api_key}

# NOAA API endpoint
BASE_URL = "https://www.ncdc.noaa.gov/cdo-web/api/v2"

# Years to download
START_YEAR = 2015
END_YEAR = 2020

# Maximum stations to collect (increase later if needed)
MAX_STATIONS = 200

all_weather = []

In [18]:
stations_url = f"{BASE_URL}/stations"

params = {
    "datasetid": "GHCND",
    "locationid": "FIPS:US",
    "limit": MAX_STATIONS
}

response = requests.get(stations_url, headers=headers, params=params)

stations = response.json()["results"]

station_ids = [s["id"] for s in stations]

print(f"Collected {len(station_ids)} stations")

Collected 200 stations


In [19]:
for station in station_ids:

    print("Downloading:", station)

    for year in range(START_YEAR, END_YEAR + 1):

        params = {
            "datasetid": "GHCND",
            "datatypeid": "TMAX",
            "stationid": station,
            "startdate": f"{year}-01-01",
            "enddate": f"{year}-12-31",
            "limit": 1000
        }

        r = requests.get(f"{BASE_URL}/data", headers=headers, params=params)

        if r.status_code != 200:
            print("Failed:", station, year)
            continue

        data = r.json()

        if "results" not in data:
            continue

        df = pd.DataFrame(data["results"])

        df["station"] = station
        df["year"] = year

        all_weather.append(df)

        time.sleep(0.3)  # prevent rate limits

Downloading: GHCND:AQC00914000
Downloading: GHCND:AQC00914005
Downloading: GHCND:AQC00914021
Downloading: GHCND:AQC00914060
Downloading: GHCND:AQC00914135
Downloading: GHCND:AQC00914138
Downloading: GHCND:AQC00914141
Downloading: GHCND:AQC00914145
Downloading: GHCND:AQC00914149
Downloading: GHCND:AQC00914188
Downloading: GHCND:AQC00914248
Downloading: GHCND:AQC00914397
Downloading: GHCND:AQC00914594
Downloading: GHCND:AQC00914650
Downloading: GHCND:AQC00914822
Failed: GHCND:AQC00914822 2016
Downloading: GHCND:AQC00914869
Downloading: GHCND:AQC00914873
Downloading: GHCND:AQC00914902
Failed: GHCND:AQC00914902 2019
Downloading: GHCND:AQC00914912
Downloading: GHCND:AQW00061705
Downloading: GHCND:CA001018611
Downloading: GHCND:CA001102420
Downloading: GHCND:CA001103635
Downloading: GHCND:CA001135126
Downloading: GHCND:CA001144230
Downloading: GHCND:CA001146000
Downloading: GHCND:CA004032335
Downloading: GHCND:CA004034912
Downloading: GHCND:CA004038116
Downloading: GHCND:CA004038740
Download

In [20]:
weather = pd.concat(all_weather, ignore_index=True)

# Convert temperature units
weather["temp_c"] = weather["value"] / 10
weather["temp_f"] = weather["temp_c"] * 9/5 + 32

# Create extreme heat label
weather["extreme_heat"] = weather["temp_f"] >= 95

# Keep useful columns
weather = weather[[
    "date",
    "station",
    "temp_f",
    "extreme_heat"
]]

In [21]:
weather.to_csv("../data/noaa_weather_dataset.csv", index=False)

print("Dataset saved.")
print("Rows:", len(weather))

Dataset saved.
Rows: 24036
